# AISC DeepFake — Validation-Weighted Soft Voting

Bu notebook, **aynı model ailesinin** göz, kaş ve ağız bölgesel tahminlerini
validation performanslarına göre ağırlıklandırarak birleştirir.

## Model families

1. **Swin V2 Tiny**
2. **EfficientNet-B0**
3. **Swin V2 Tiny + Texture Fusion**

## Weighted Soft Voting

Her bölgenin ağırlığı yalnızca o bölgenin **validation ROC-AUC** değerinden
türetilir:

\[
r_i = \max(AUC_i - 0.5, \varepsilon)
\]

\[
w_i = \frac{r_i}{\sum_j r_j}
\]

\[
p_{fusion}
=
w_{eye}p_{eye}
+
w_{brow}p_{brow}
+
w_{mouth}p_{mouth}
\]

## Critical scientific rules

- **Test set is never used to learn weights.**
- **Test set is never used to tune the decision threshold.**
- Decision threshold is fixed in advance at **0.50**.
- Validation ROC-AUC values are recovered from each experiment's saved
  validation/training metric artifacts.
- If validation ROC-AUC cannot be verified, that model family **fails fast**.
  The notebook does not silently fall back to test metrics.
- Eye, brow and mouth predictions are aligned by the same upstream frame.
- Only the three-region frame intersection is evaluated.
- Source data, ROI files, checkpoints and prediction CSV files are read-only.
  This notebook never deletes, renames or overwrites them.

## Implemented project standards

Bu sürümde:

- reproducibility (`seed=42`);
- read-only source data;
- explicit run ID;
- fail-fast schema and numerical quality gates;
- traceable `sample_id → metadata → source_frame` recovery;
- validation-only weight estimation;
- atomic CSV/JSON output writes;
- per-family audit logs;
- no `except: pass`;
- English figures;
- PNG **600 DPI** + SVG outputs;
- minimum figure pixel-size verification;
- output manifest and final quality gates

uygulanmıştır.

In [1]:
# ============================================================
# 1) COLAB + CONFIG
# ============================================================

from google.colab import drive

drive.mount("/content/drive")

from pathlib import Path
from datetime import datetime, timezone
from importlib.metadata import PackageNotFoundError, version

import json
import math
import os
import platform
import re
import sys
import warnings

import numpy as np
import pandas as pd

SEED = 42
np.random.seed(SEED)

METHOD_NAME = "02_weighted_soft_voting"
DECISION_THRESHOLD = 0.50
FRAME_AGGREGATION = "mean"
FIGURE_DPI = 600
MIN_FIGURE_SHORT_EDGE_PX = 600
WEIGHT_EPSILON = 1e-6

OUTPUT_ROOT = Path(
    "/content/drive/MyDrive/"
    "AISC DeepFake Çalışmaları/Deney 1/"
    "Kader/Deney 1/Sonuçlar/Fusion_Experiments"
)

OUTPUT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

SEARCH_ROOTS = {
    "eye": Path(
        "/content/drive/MyDrive/"
        "AISC DeepFake Çalışmaları/Deney 1/"
        "Kader/Deney 1/Sonuçlar"
    ),
    "brow": Path(
        "/content/drive/MyDrive/"
        "AISC DeepFake Çalışmaları/Deney 1/"
        "Nazlıcan/Deney 1/Sonuçlar"
    ),
    "mouth": Path(
        "/content/drive/MyDrive/"
        "AISC DeepFake Çalışmaları/Deney 1/"
        "Dilara/Deney 1/Sonuçlar"
    ),
}

# ------------------------------------------------------------------
# Verified experiment result paths used in the previous fusion work.
# These source prediction files are READ ONLY.
# ------------------------------------------------------------------

MODEL_FAMILIES = {
    "swinv2_tiny": {
        "eye_test": (
            SEARCH_ROOTS["eye"]
            / "20260807_1031_eye_swinv2_tiny_seed42"
            / "predictions"
            / "test_frame_predictions.csv"
        ),
        "brow_test": (
            SEARCH_ROOTS["brow"]
            / "Kas_SwinV2_Tiny_Detayli_Sonuc_pdf"
            / "predictions"
            / "test_frame_predictions.csv"
        ),
        "mouth_test": (
            SEARCH_ROOTS["mouth"]
            / "20260807_2235_mouth_swinv2_tiny_seed42"
            / "predictions"
            / "test_frame_predictions.csv"
        ),
    },

    "efficientnet_b0": {
        "eye_test": (
            SEARCH_ROOTS["eye"]
            / "20260808_0803_eye_efficientnet_b0_seed42"
            / "predictions"
            / "test_predictions.csv"
        ),
        "brow_test": (
            SEARCH_ROOTS["brow"]
            / "20260808_1248_eyebrow_efficientnet_b0_seed42"
            / "predictions"
            / "test_predictions.csv"
        ),
        "mouth_test": (
            SEARCH_ROOTS["mouth"]
            / "20260808_1257_mouth_efficientnet_b0_seed42"
            / "predictions"
            / "test_predictions.csv"
        ),
    },

    "swinv2_texture": {
        "eye_test": (
            SEARCH_ROOTS["eye"]
            / "20260806_1748_eye_swinv2_texturefusion_seed42"
            / "full"
            / "predictions"
            / "test_predictions.csv"
        ),
        "brow_test": (
            SEARCH_ROOTS["brow"]
            / "Swin V2-Tiny + LBP + GLCM + Gabor + Wavelet Fusion"
            / "predictions"
            / "test_predictions_frame_level.csv"
        ),
        "mouth_test": (
            SEARCH_ROOTS["mouth"]
            / "SwinV2_TextureFusion_Mouth"
            / "20260807_1550_mouth_swinv2_texturefusion_seed42"
            / "full"
            / "predictions"
            / "test_predictions.csv"
        ),
    },
}

# Optional validation-AUC overrides.
# Leave as None to use automatic, auditable recovery from saved validation metrics.
# Never place TEST metrics here.
VALIDATION_AUC_OVERRIDE = {
    family: {
        "eye": None,
        "brow": None,
        "mouth": None,
    }
    for family in MODEL_FAMILIES
}

RUN_ID = (
    datetime.now(timezone.utc)
    .strftime("%Y%m%d_%H%M%S_%f")
    + "_weighted_soft_voting_seed42"
)

RUN_DIR = (
    OUTPUT_ROOT
    / METHOD_NAME
    / RUN_ID
)

RUN_DIR.mkdir(
    parents=True,
    exist_ok=False,
)

print(f"Run ID     : {RUN_ID}")
print(f"Run output : {RUN_DIR}")

Mounted at /content/drive
Run ID     : 20260809_163157_934154_weighted_soft_voting_seed42
Run output : /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deney 1/Kader/Deney 1/Sonuçlar/Fusion_Experiments/02_weighted_soft_voting/20260809_163157_934154_weighted_soft_voting_seed42


In [2]:
# ============================================================
# 2) ENVIRONMENT + ATOMIC I/O
# ============================================================

import matplotlib
import matplotlib.pyplot as plt

from PIL import Image


def package_version(package_name):
    try:
        return version(package_name)
    except PackageNotFoundError:
        return "not_installed"


def json_default(value):
    if isinstance(value, np.integer):
        return int(value)

    if isinstance(value, np.floating):
        return float(value)

    if isinstance(value, np.ndarray):
        return value.tolist()

    if isinstance(value, Path):
        return str(value)

    raise TypeError(
        f"Object of type {type(value).__name__} "
        "is not JSON serializable."
    )


def atomic_write_json(payload, target):
    target = Path(target)
    target.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temp_path = target.with_suffix(
        target.suffix + ".tmp"
    )

    with temp_path.open(
        "w",
        encoding="utf-8",
    ) as file_handle:
        json.dump(
            payload,
            file_handle,
            indent=2,
            ensure_ascii=False,
            default=json_default,
        )
        file_handle.flush()
        os.fsync(
            file_handle.fileno()
        )

    with temp_path.open(
        "r",
        encoding="utf-8",
    ) as file_handle:
        json.load(file_handle)

    os.replace(
        temp_path,
        target,
    )


def atomic_write_csv(dataframe, target):
    target = Path(target)
    target.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temp_path = target.with_suffix(
        target.suffix + ".tmp"
    )

    dataframe.to_csv(
        temp_path,
        index=False,
    )

    verification = pd.read_csv(
        temp_path
    )

    if len(verification) != len(dataframe):
        raise RuntimeError(
            f"Atomic CSV verification failed for {target}. "
            f"Expected {len(dataframe)} rows, "
            f"read back {len(verification)}."
        )

    os.replace(
        temp_path,
        target,
    )


def save_figure(fig, target_stem):
    target_stem = Path(target_stem)

    target_stem.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    png_path = target_stem.with_suffix(
        ".png"
    )
    svg_path = target_stem.with_suffix(
        ".svg"
    )

    temp_png = png_path.with_suffix(
        ".png.tmp"
    )
    temp_svg = svg_path.with_suffix(
        ".svg.tmp"
    )

    fig.savefig(
        temp_png,
        format="png",
        dpi=FIGURE_DPI,
        bbox_inches="tight",
    )

    fig.savefig(
        temp_svg,
        format="svg",
        bbox_inches="tight",
    )

    with Image.open(
        temp_png
    ) as image:
        if min(image.size) < MIN_FIGURE_SHORT_EDGE_PX:
            raise RuntimeError(
                "Figure resolution is insufficient: "
                f"{image.size} -> {png_path}"
            )

    os.replace(
        temp_png,
        png_path,
    )

    os.replace(
        temp_svg,
        svg_path,
    )

    plt.close(fig)

    return png_path, svg_path


ENVIRONMENT = {
    "run_id": RUN_ID,
    "created_at_utc": (
        datetime.now(timezone.utc)
        .isoformat()
    ),
    "python": sys.version,
    "platform": platform.platform(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "matplotlib": matplotlib.__version__,
    "scikit_learn": package_version(
        "scikit-learn"
    ),
    "pillow": package_version(
        "Pillow"
    ),
    "seed": SEED,
    "method": METHOD_NAME,
    "decision_threshold": DECISION_THRESHOLD,
    "frame_aggregation": FRAME_AGGREGATION,
    "figure_dpi": FIGURE_DPI,
    "weight_formula": (
        "max(validation_roc_auc - 0.5, epsilon), "
        "then normalize to sum=1"
    ),
}

atomic_write_json(
    ENVIRONMENT,
    RUN_DIR / "environment.json",
)

In [3]:
# ============================================================
# 3) PREDICTION SCHEMA + FRAME ALIGNMENT
# ============================================================

from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)

LABEL_CANDIDATES = [
    "true_label",
    "label",
    "label_int",
    "target",
    "y_true",
    "ground_truth",
    "class_id",
    "true_class",
]

PROB_CANDIDATES = [
    "fake_probability",
    "prob_fake",
    "probability_fake",
    "probability",
    "prob",
    "y_score",
    "score",
    "fake_prob",
    "prediction_probability",
]

SOURCE_KEY_CANDIDATES = [
    "source_frame",
    "relative_frame_path",
    "input_path",
    "input_relative_path",
    "original_frame",
    "orijinal_yol",
    "frame_stem",
    "image_path",
    "path",
]


def first_existing(columns, candidates):
    lower_to_original = {
        str(column).lower(): column
        for column in columns
    }

    for candidate in candidates:
        if candidate.lower() in lower_to_original:
            return lower_to_original[
                candidate.lower()
            ]

    return None


def normalize_label(value):
    if pd.isna(value):
        return np.nan

    if isinstance(
        value,
        (
            int,
            np.integer,
            float,
            np.floating,
        ),
    ):
        return int(
            float(value) >= 0.5
        )

    normalized = (
        str(value)
        .strip()
        .lower()
    )

    fake_values = {
        "1",
        "fake",
        "deepfake",
        "manipulated",
        "sahte",
        "f",
    }

    real_values = {
        "0",
        "real",
        "original",
        "genuine",
        "gerçek",
        "r",
    }

    if normalized in fake_values:
        return 1

    if normalized in real_values:
        return 0

    try:
        return int(
            float(normalized) >= 0.5
        )
    except (
        TypeError,
        ValueError,
    ) as exc:
        raise ValueError(
            f"Label could not be normalized: {value!r}"
        ) from exc


def canonical_frame_key(value):
    """
    Normalize region-specific ROI names to one upstream frame key.

    Examples
    --------
    fake_test_00000__face_00.jpg
    fake_test_00000.jpg
    fake_test_fake_test_00000_face00.png

    -> fake_test_00000

    Unknown formats return None.
    """

    if pd.isna(value):
        return None

    text = (
        str(value)
        .replace("\\", "/")
        .strip()
        .lower()
    )

    filename = text.split("/")[-1]

    filename = re.sub(
        r"\.(jpg|jpeg|png|bmp|webp|npy)$",
        "",
        filename,
        flags=re.IGNORECASE,
    )

    matches = re.findall(
        r"(real|fake)_(train|test|val|validation)_(\d+)",
        filename,
        flags=re.IGNORECASE,
    )

    if not matches:
        return None

    label, split, frame_number = matches[-1]

    split = split.lower()

    if split == "validation":
        split = "val"

    frame_number = (
        str(int(frame_number))
        .zfill(5)
    )

    return (
        f"{label.lower()}_"
        f"{split}_"
        f"{frame_number}"
    )


def find_companion_metadata(
    prediction_path,
):
    """
    Find metadata that maps cache sample_id values back to source frames.

    Search is limited to the prediction file's own experiment ancestry.
    No source file is modified.
    """

    prediction_path = Path(
        prediction_path
    )

    metadata_names = [
        "eligible_metadata.csv",
        "eligible_metadata_before_cache.csv",
        "metadata_used.csv",
    ]

    candidate_paths = []

    current = prediction_path.parent

    for _ in range(6):
        for metadata_name in metadata_names:
            candidate_paths.append(
                current
                / "artifacts"
                / metadata_name
            )

            candidate_paths.append(
                current
                / metadata_name
            )

        if current.parent == current:
            break

        current = current.parent

    for candidate in candidate_paths:
        if candidate.is_file():
            return candidate

    return None


def build_key_from_metadata(
    prediction_df,
    prediction_path,
    region_name,
):
    if "sample_id" not in prediction_df.columns:
        raise ValueError(
            f"{region_name}: direct frame key is unavailable "
            "and sample_id is missing."
        )

    metadata_path = find_companion_metadata(
        prediction_path
    )

    if metadata_path is None:
        raise FileNotFoundError(
            f"{region_name}: cache/hash prediction path detected, "
            "but no companion metadata was found.\n"
            f"Prediction: {prediction_path}"
        )

    metadata = pd.read_csv(
        metadata_path
    )

    if "sample_id" not in metadata.columns:
        raise ValueError(
            f"{region_name}: companion metadata does not contain sample_id.\n"
            f"Metadata: {metadata_path}"
        )

    source_column = None
    source_keys = None

    for candidate in SOURCE_KEY_CANDIDATES:
        if candidate not in metadata.columns:
            continue

        candidate_keys = (
            metadata[candidate]
            .map(canonical_frame_key)
        )

        valid_fraction = float(
            candidate_keys
            .notna()
            .mean()
        )

        if valid_fraction >= 0.95:
            source_column = candidate
            source_keys = candidate_keys
            break

    if source_column is None:
        raise ValueError(
            f"{region_name}: no trustworthy upstream-frame column "
            "was found in companion metadata.\n"
            f"Metadata columns: {list(metadata.columns)}"
        )

    mapping = pd.DataFrame(
        {
            "sample_id": (
                metadata["sample_id"]
                .astype(str)
                .str.strip()
            ),
            "fusion_key": source_keys,
        }
    )

    mapping = (
        mapping
        .dropna(
            subset=["fusion_key"]
        )
        .drop_duplicates(
            subset=["sample_id"]
        )
    )

    prediction_ids = (
        prediction_df["sample_id"]
        .astype(str)
        .str.strip()
    )

    temporary = pd.DataFrame(
        {
            "sample_id": prediction_ids,
        }
    )

    temporary = temporary.merge(
        mapping,
        on="sample_id",
        how="left",
        validate="many_to_one",
    )

    missing_count = int(
        temporary["fusion_key"]
        .isna()
        .sum()
    )

    if missing_count:
        examples = (
            temporary.loc[
                temporary["fusion_key"].isna(),
                "sample_id",
            ]
            .head(10)
            .tolist()
        )

        raise ValueError(
            f"{region_name}: {missing_count} prediction sample_id values "
            "could not be mapped to source frames.\n"
            f"Examples: {examples}"
        )

    audit = {
        "key_resolution": "companion_metadata",
        "metadata_path": str(
            metadata_path
        ),
        "metadata_source_column": str(
            source_column
        ),
    }

    return (
        temporary["fusion_key"],
        audit,
    )


def resolve_fusion_key(
    raw,
    prediction_path,
    region_name,
):
    direct_candidates = [
        "source_frame",
        "relative_frame_path",
        "frame_path",
        "original_frame",
        "image_path",
        "path",
        "frame_stem",
    ]

    for column in direct_candidates:
        if column not in raw.columns:
            continue

        keys = (
            raw[column]
            .map(canonical_frame_key)
        )

        valid_fraction = float(
            keys
            .notna()
            .mean()
        )

        if valid_fraction >= 0.95:
            return keys, {
                "key_resolution": "prediction_column",
                "key_column": str(column),
            }

    return build_key_from_metadata(
        prediction_df=raw,
        prediction_path=prediction_path,
        region_name=region_name,
    )


def load_prediction_csv(
    path,
    region_name,
):
    if path is None:
        raise FileNotFoundError(
            f"{region_name}: prediction CSV path is None."
        )

    path = Path(path)

    if not path.is_file():
        raise FileNotFoundError(
            f"{region_name}: prediction CSV does not exist:\n{path}"
        )

    raw = pd.read_csv(
        path
    )

    if raw.empty:
        raise ValueError(
            f"{region_name}: prediction CSV is empty:\n{path}"
        )

    label_col = first_existing(
        raw.columns,
        LABEL_CANDIDATES,
    )

    prob_col = first_existing(
        raw.columns,
        PROB_CANDIDATES,
    )

    if label_col is None:
        raise ValueError(
            f"{region_name}: label column not found.\n"
            f"Columns: {list(raw.columns)}"
        )

    if prob_col is None:
        raise ValueError(
            f"{region_name}: fake probability column not found.\n"
            f"Columns: {list(raw.columns)}"
        )

    normalized_labels = (
        raw[label_col]
        .map(normalize_label)
    )

    if normalized_labels.isna().any():
        raise ValueError(
            f"{region_name}: invalid label values found."
        )

    probabilities = pd.to_numeric(
        raw[prob_col],
        errors="coerce",
    )

    if probabilities.isna().any():
        raise ValueError(
            f"{region_name}: invalid probability values found."
        )

    probability_out_of_range = (
        (probabilities < 0)
        | (probabilities > 1)
    )

    if probability_out_of_range.any():
        raise ValueError(
            f"{region_name}: "
            f"{int(probability_out_of_range.sum())} probabilities "
            "are outside [0, 1]."
        )

    fusion_keys, key_audit = (
        resolve_fusion_key(
            raw=raw,
            prediction_path=path,
            region_name=region_name,
        )
    )

    if fusion_keys.isna().any():
        raise ValueError(
            f"{region_name}: unresolved fusion keys remain."
        )

    if "sample_id" in raw.columns:
        raw_identifier = (
            raw["sample_id"]
            .astype(str)
        )
    elif "image_path" in raw.columns:
        raw_identifier = (
            raw["image_path"]
            .astype(str)
        )
    elif "path" in raw.columns:
        raw_identifier = (
            raw["path"]
            .astype(str)
        )
    else:
        raw_identifier = (
            pd.Series(
                np.arange(len(raw)),
                index=raw.index,
            )
            .astype(str)
        )

    normalized = pd.DataFrame(
        {
            "fusion_key": fusion_keys,
            f"label_{region_name}": (
                normalized_labels
                .astype(int)
            ),
            f"p_{region_name}": (
                probabilities
                .astype(float)
            ),
            f"raw_key_{region_name}": raw_identifier,
        }
    )

    grouped = (
        normalized
        .groupby(
            "fusion_key",
            as_index=False,
        )
        .agg(
            **{
                f"label_min_{region_name}": (
                    f"label_{region_name}",
                    "min",
                ),
                f"label_max_{region_name}": (
                    f"label_{region_name}",
                    "max",
                ),
                f"p_{region_name}": (
                    f"p_{region_name}",
                    FRAME_AGGREGATION,
                ),
                f"raw_key_{region_name}": (
                    f"raw_key_{region_name}",
                    "first",
                ),
                f"roi_count_{region_name}": (
                    f"p_{region_name}",
                    "size",
                ),
            }
        )
    )

    inconsistent = (
        grouped[
            f"label_min_{region_name}"
        ]
        != grouped[
            f"label_max_{region_name}"
        ]
    )

    if inconsistent.any():
        raise ValueError(
            f"{region_name}: "
            f"{int(inconsistent.sum())} upstream frames "
            "contain conflicting labels."
        )

    grouped[
        f"label_{region_name}"
    ] = (
        grouped[
            f"label_min_{region_name}"
        ]
        .astype(int)
    )

    grouped = grouped.drop(
        columns=[
            f"label_min_{region_name}",
            f"label_max_{region_name}",
        ]
    )

    audit = {
        "region": region_name,
        "prediction_path": str(path),
        "source_rows": int(len(raw)),
        "unique_upstream_frames": int(
            len(grouped)
        ),
        "multi_roi_frames": int(
            (
                grouped[
                    f"roi_count_{region_name}"
                ]
                > 1
            )
            .sum()
        ),
        "max_roi_count_per_frame": int(
            grouped[
                f"roi_count_{region_name}"
            ]
            .max()
        ),
        "label_column": str(label_col),
        "probability_column": str(prob_col),
        "frame_aggregation": FRAME_AGGREGATION,
        **key_audit,
    }

    print(
        f"{region_name.upper():5s} | "
        f"Rows={len(raw)} | "
        f"Frames={len(grouped)} | "
        f"Key={key_audit['key_resolution']}"
    )

    return grouped, audit


def align_three(
    eye_path,
    brow_path,
    mouth_path,
):
    eye, eye_audit = (
        load_prediction_csv(
            eye_path,
            "eye",
        )
    )

    brow, brow_audit = (
        load_prediction_csv(
            brow_path,
            "brow",
        )
    )

    mouth, mouth_audit = (
        load_prediction_csv(
            mouth_path,
            "mouth",
        )
    )

    aligned = eye.merge(
        brow,
        on="fusion_key",
        how="inner",
        validate="one_to_one",
    )

    aligned = aligned.merge(
        mouth,
        on="fusion_key",
        how="inner",
        validate="one_to_one",
    )

    if aligned.empty:
        raise RuntimeError(
            "Eye/Brow/Mouth upstream frame intersection is empty."
        )

    label_columns = [
        "label_eye",
        "label_brow",
        "label_mouth",
    ]

    labels = (
        aligned[label_columns]
        .astype(int)
    )

    consistent = (
        labels
        .nunique(axis=1)
        .eq(1)
    )

    if not consistent.all():
        examples = (
            aligned.loc[
                ~consistent,
                ["fusion_key"] + label_columns,
            ]
            .head(10)
        )

        raise ValueError(
            f"{int((~consistent).sum())} aligned frames have "
            f"inconsistent region labels.\n{examples}"
        )

    # IMPORTANT:
    # Create final label only AFTER checking the three source labels.
    # Do not drop it afterwards.
    aligned["label"] = (
        labels.iloc[:, 0]
        .astype(int)
    )

    for column in [
        "p_eye",
        "p_brow",
        "p_mouth",
    ]:
        values = pd.to_numeric(
            aligned[column],
            errors="coerce",
        )

        if values.isna().any():
            raise ValueError(
                f"{column}: NaN probability after alignment."
            )

        if (
            (values < 0)
            | (values > 1)
        ).any():
            raise ValueError(
                f"{column}: probability outside [0, 1]."
            )

        aligned[column] = (
            values.astype(float)
        )

    aligned = (
        aligned
        .sort_values("fusion_key")
        .reset_index(drop=True)
    )

    counts = {
        "eye_frames": int(len(eye)),
        "brow_frames": int(len(brow)),
        "mouth_frames": int(len(mouth)),
        "common_frames": int(
            len(aligned)
        ),
        "real_common_frames": int(
            (aligned["label"] == 0)
            .sum()
        ),
        "fake_common_frames": int(
            (aligned["label"] == 1)
            .sum()
        ),
    }

    if (
        counts["real_common_frames"] == 0
        or counts["fake_common_frames"] == 0
    ):
        raise RuntimeError(
            "Aligned common set must contain both REAL and FAKE."
        )

    audit = {
        "counts": counts,
        "eye": eye_audit,
        "brow": brow_audit,
        "mouth": mouth_audit,
    }

    return aligned, audit


def compute_metrics(
    y_true,
    probabilities,
    threshold=DECISION_THRESHOLD,
):
    y_true = np.asarray(
        y_true,
        dtype=int,
    )

    probabilities = np.asarray(
        probabilities,
        dtype=float,
    )

    if len(y_true) != len(probabilities):
        raise ValueError(
            "y_true and probabilities have different lengths."
        )

    if not np.isfinite(
        probabilities
    ).all():
        raise ValueError(
            "NaN/Inf found in probabilities."
        )

    predicted = (
        probabilities >= threshold
    ).astype(int)

    tn, fp, fn, tp = (
        confusion_matrix(
            y_true,
            predicted,
            labels=[0, 1],
        )
        .ravel()
    )

    return {
        "n": int(len(y_true)),
        "threshold": float(threshold),
        "accuracy": float(
            accuracy_score(
                y_true,
                predicted,
            )
        ),
        "balanced_accuracy": float(
            balanced_accuracy_score(
                y_true,
                predicted,
            )
        ),
        "precision": float(
            precision_score(
                y_true,
                predicted,
                zero_division=0,
            )
        ),
        "recall": float(
            recall_score(
                y_true,
                predicted,
                zero_division=0,
            )
        ),
        "specificity": float(
            tn / (tn + fp)
            if (tn + fp)
            else np.nan
        ),
        "f1": float(
            f1_score(
                y_true,
                predicted,
                zero_division=0,
            )
        ),
        "roc_auc": float(
            roc_auc_score(
                y_true,
                probabilities,
            )
        ),
        "pr_auc": float(
            average_precision_score(
                y_true,
                probabilities,
            )
        ),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }

In [4]:
# ============================================================
# 4) VALIDATION ROC-AUC RECOVERY — NO TEST LEAKAGE
# ============================================================

def is_validation_auc_name(name):
    normalized = (
        str(name)
        .strip()
        .lower()
        .replace("-", "_")
        .replace(" ", "_")
    )

    has_validation = (
        "val" in normalized
        or "validation" in normalized
    )

    has_auc = (
        "auc" in normalized
    )

    return (
        has_validation
        and has_auc
    )


def candidate_experiment_roots(
    prediction_path,
):
    prediction_path = Path(
        prediction_path
    )

    roots = []
    current = prediction_path.parent

    for _ in range(6):
        roots.append(current)

        if current.parent == current:
            break

        current = current.parent

    # Preserve order, remove duplicates.
    unique_roots = []

    for root in roots:
        if root not in unique_roots:
            unique_roots.append(root)

    return unique_roots


def flatten_json(
    payload,
    prefix="",
):
    flattened = {}

    if isinstance(payload, dict):
        for key, value in payload.items():
            child_key = (
                f"{prefix}.{key}"
                if prefix
                else str(key)
            )

            flattened.update(
                flatten_json(
                    value,
                    child_key,
                )
            )

    elif isinstance(payload, list):
        for index, value in enumerate(payload):
            child_key = (
                f"{prefix}[{index}]"
            )

            flattened.update(
                flatten_json(
                    value,
                    child_key,
                )
            )

    else:
        flattened[prefix] = payload

    return flattened


def collect_validation_auc_candidates(
    prediction_path,
):
    """
    Search saved experiment metrics only.

    Accepted sources:
    - JSON summary/metric files with val/validation + AUC keys.
    - CSV history/metric files with val/validation + AUC columns.

    Prediction CSVs and TEST metric values are never used as weight sources.
    """

    prediction_path = Path(
        prediction_path
    )

    candidates = []
    visited = set()

    roots = candidate_experiment_roots(
        prediction_path
    )

    # Keep search bounded to metric-like files.
    filename_tokens = (
        "history",
        "metric",
        "summary",
        "result",
    )

    for root_index, root in enumerate(roots):
        if not root.exists():
            continue

        files = []

        for suffix in (
            "*.json",
            "*.csv",
        ):
            try:
                files.extend(
                    root.rglob(suffix)
                )
            except OSError:
                continue

        for file_path in files:
            file_path = Path(
                file_path
            )

            if file_path in visited:
                continue

            visited.add(
                file_path
            )

            # Never derive validation weights from prediction files.
            lower_name = (
                file_path.name
                .lower()
            )

            if "pred" in lower_name:
                continue

            if not any(
                token in lower_name
                for token in filename_tokens
            ):
                continue

            # Avoid crawling unrelated nested fusion outputs.
            if (
                "fusion_experiments"
                in str(file_path).lower()
            ):
                continue

            try:
                if file_path.suffix.lower() == ".json":
                    with file_path.open(
                        "r",
                        encoding="utf-8",
                    ) as file_handle:
                        payload = json.load(
                            file_handle
                        )

                    flattened = flatten_json(
                        payload
                    )

                    for key, value in flattened.items():
                        if not is_validation_auc_name(
                            key
                        ):
                            continue

                        try:
                            numeric_value = float(
                                value
                            )
                        except (
                            TypeError,
                            ValueError,
                        ):
                            continue

                        if not (
                            0.0
                            <= numeric_value
                            <= 1.0
                        ):
                            continue

                        key_lower = key.lower()

                        priority = 80

                        if "best" in key_lower:
                            priority += 20

                        if "roc" in key_lower:
                            priority += 10

                        if "summary" in lower_name:
                            priority += 5

                        # Shallower experiment root is preferred.
                        priority -= root_index

                        candidates.append(
                            {
                                "auc": numeric_value,
                                "source_path": str(
                                    file_path
                                ),
                                "source_field": str(
                                    key
                                ),
                                "source_type": "json",
                                "priority": int(
                                    priority
                                ),
                                "selection": "explicit_saved_value",
                            }
                        )

                elif file_path.suffix.lower() == ".csv":
                    dataframe = pd.read_csv(
                        file_path
                    )

                    if dataframe.empty:
                        continue

                    for column in dataframe.columns:
                        if not is_validation_auc_name(
                            column
                        ):
                            continue

                        numeric = pd.to_numeric(
                            dataframe[column],
                            errors="coerce",
                        ).dropna()

                        numeric = numeric[
                            (numeric >= 0.0)
                            & (numeric <= 1.0)
                        ]

                        if numeric.empty:
                            continue

                        column_lower = (
                            str(column)
                            .lower()
                        )

                        # Training history is expected to contain one
                        # validation AUC per epoch. The maximum corresponds
                        # to the best observed validation discriminator.
                        numeric_value = float(
                            numeric.max()
                        )

                        priority = 60

                        if "best" in column_lower:
                            priority += 20

                        if "roc" in column_lower:
                            priority += 10

                        if "history" in lower_name:
                            priority += 5

                        priority -= root_index

                        candidates.append(
                            {
                                "auc": numeric_value,
                                "source_path": str(
                                    file_path
                                ),
                                "source_field": str(
                                    column
                                ),
                                "source_type": "csv",
                                "priority": int(
                                    priority
                                ),
                                "selection": "max_validation_auc_in_column",
                            }
                        )

            except (
                OSError,
                UnicodeDecodeError,
                json.JSONDecodeError,
                pd.errors.ParserError,
                pd.errors.EmptyDataError,
            ) as exc:
                warnings.warn(
                    f"Metric file skipped because it could not be read: "
                    f"{file_path} | {exc}"
                )

    return candidates


def resolve_validation_auc(
    family,
    region,
    prediction_path,
):
    override = (
        VALIDATION_AUC_OVERRIDE[
            family
        ][region]
    )

    if override is not None:
        override = float(
            override
        )

        if not (
            0.0
            <= override
            <= 1.0
        ):
            raise ValueError(
                f"{family}/{region}: validation AUC override "
                "must be within [0, 1]."
            )

        return {
            "auc": override,
            "source_path": "manual_override",
            "source_field": (
                f"VALIDATION_AUC_OVERRIDE[{family}][{region}]"
            ),
            "source_type": "manual_override",
            "priority": 1000,
            "selection": "manual_validation_auc_override",
        }

    candidates = (
        collect_validation_auc_candidates(
            prediction_path
        )
    )

    if not candidates:
        raise RuntimeError(
            f"{family}/{region}: no verifiable validation ROC-AUC "
            "could be recovered from saved experiment metrics.\n"
            "Weighted fusion will NOT fall back to TEST metrics.\n"
            "If you have a verified validation ROC-AUC, place it in "
            "VALIDATION_AUC_OVERRIDE."
        )

    candidates = sorted(
        candidates,
        key=lambda item: (
            item["priority"],
            item["auc"],
        ),
        reverse=True,
    )

    selected = candidates[0]

    selected = {
        **selected,
        "candidate_count": int(
            len(candidates)
        ),
        "top_candidates": candidates[:10],
    }

    return selected


def compute_validation_weights(
    family,
    cfg,
):
    validation_records = {}

    for region in (
        "eye",
        "brow",
        "mouth",
    ):
        validation_records[region] = (
            resolve_validation_auc(
                family=family,
                region=region,
                prediction_path=cfg[
                    f"{region}_test"
                ],
            )
        )

    validation_auc = {
        region: float(
            record["auc"]
        )
        for region, record
        in validation_records.items()
    }

    raw_scores = {
        region: max(
            auc_value - 0.5,
            WEIGHT_EPSILON,
        )
        for region, auc_value
        in validation_auc.items()
    }

    denominator = float(
        sum(
            raw_scores.values()
        )
    )

    if (
        not np.isfinite(
            denominator
        )
        or denominator <= 0
    ):
        raise RuntimeError(
            f"{family}: invalid weight denominator."
        )

    weights = {
        region: float(
            raw_score
            / denominator
        )
        for region, raw_score
        in raw_scores.items()
    }

    if not math.isclose(
        sum(weights.values()),
        1.0,
        rel_tol=1e-9,
        abs_tol=1e-9,
    ):
        raise RuntimeError(
            f"{family}: normalized weights do not sum to 1."
        )

    return {
        "validation_auc": validation_auc,
        "raw_weight_scores": raw_scores,
        "weights": weights,
        "validation_sources": validation_records,
    }

In [5]:
# ============================================================
# 5) PRE-FLIGHT QUALITY GATES
# ============================================================

EXPECTED_KEY_CASES = {
    "fake_test_00000__face_00.jpg": "fake_test_00000",
    "fake_test_00000.jpg": "fake_test_00000",
    "fake_test_fake_test_00000_face00.png": "fake_test_00000",
    "real_test_00125__face_00.jpg": "real_test_00125",
    "real_test_00125.jpg": "real_test_00125",
    "real_test_real_test_00125_face00.png": "real_test_00125",
}

for raw_key, expected_key in EXPECTED_KEY_CASES.items():
    actual_key = canonical_frame_key(
        raw_key
    )

    if actual_key != expected_key:
        raise RuntimeError(
            f"Frame-key quality gate failed: "
            f"{raw_key} -> {actual_key}; "
            f"expected {expected_key}"
        )

print("Frame-key normalization: PASSED")


path_rows = []

for family, cfg in MODEL_FAMILIES.items():
    for region in (
        "eye",
        "brow",
        "mouth",
    ):
        path = Path(
            cfg[f"{region}_test"]
        )

        path_rows.append(
            {
                "model_family": family,
                "region": region,
                "path": str(path),
                "exists": bool(
                    path.is_file()
                ),
            }
        )

path_audit = pd.DataFrame(
    path_rows
)

if not path_audit[
    "exists"
].all():
    missing = path_audit.loc[
        ~path_audit["exists"],
        [
            "model_family",
            "region",
            "path",
        ],
    ]

    raise FileNotFoundError(
        "Configured prediction file(s) missing:\n"
        + missing.to_string(
            index=False
        )
    )

print("Prediction paths: PASSED")

atomic_write_csv(
    path_audit,
    RUN_DIR
    / "audit"
    / "configured_prediction_paths.csv",
)

display(
    path_audit
)

Frame-key normalization: PASSED
Prediction paths: PASSED


,model_family,region,path,exists
0,swinv2_tiny,eye,/content/drive/MyDrive/AISC DeepFake Çalışmala...,True
1,swinv2_tiny,brow,/content/drive/MyDrive/AISC DeepFake Çalışmala...,True
2,swinv2_tiny,mouth,/content/drive/MyDrive/AISC DeepFake Çalışmala...,True
3,efficientnet_b0,eye,/content/drive/MyDrive/AISC DeepFake Çalışmala...,True
4,efficientnet_b0,brow,/content/drive/MyDrive/AISC DeepFake Çalışmala...,True
5,efficientnet_b0,mouth,/content/drive/MyDrive/AISC DeepFake Çalışmala...,True
6,swinv2_texture,eye,/content/drive/MyDrive/AISC DeepFake Çalışmala...,True
7,swinv2_texture,brow,/content/drive/MyDrive/AISC DeepFake Çalışmala...,True
8,swinv2_texture,mouth,/content/drive/MyDrive/AISC DeepFake Çalışmala...,True


In [6]:
# ============================================================
# 6) VALIDATION-WEIGHT AUDIT
# ============================================================

weight_audit_rows = []
WEIGHT_CONFIGS = {}

for family, cfg in MODEL_FAMILIES.items():
    print("\n" + "=" * 90)
    print(
        f"VALIDATION WEIGHT RECOVERY: {family}"
    )
    print("=" * 90)

    weight_config = (
        compute_validation_weights(
            family=family,
            cfg=cfg,
        )
    )

    WEIGHT_CONFIGS[
        family
    ] = weight_config

    for region in (
        "eye",
        "brow",
        "mouth",
    ):
        source = (
            weight_config[
                "validation_sources"
            ][region]
        )

        row = {
            "model_family": family,
            "region": region,
            "validation_roc_auc": (
                weight_config[
                    "validation_auc"
                ][region]
            ),
            "raw_weight_score": (
                weight_config[
                    "raw_weight_scores"
                ][region]
            ),
            "normalized_weight": (
                weight_config[
                    "weights"
                ][region]
            ),
            "source_path": (
                source[
                    "source_path"
                ]
            ),
            "source_field": (
                source[
                    "source_field"
                ]
            ),
            "source_type": (
                source[
                    "source_type"
                ]
            ),
            "selection": (
                source[
                    "selection"
                ]
            ),
        }

        weight_audit_rows.append(
            row
        )

    print(
        "Validation ROC-AUC:",
        weight_config[
            "validation_auc"
        ],
    )

    print(
        "Normalized weights:",
        weight_config[
            "weights"
        ],
    )


weight_audit = pd.DataFrame(
    weight_audit_rows
)

atomic_write_csv(
    weight_audit,
    RUN_DIR
    / "audit"
    / "validation_weight_sources.csv",
)

atomic_write_json(
    WEIGHT_CONFIGS,
    RUN_DIR
    / "audit"
    / "validation_weight_config.json",
)

display(
    weight_audit[
        [
            "model_family",
            "region",
            "validation_roc_auc",
            "normalized_weight",
            "source_path",
            "source_field",
        ]
    ]
)


VALIDATION WEIGHT RECOVERY: swinv2_tiny
Validation ROC-AUC: {'eye': 0.7110501029512697, 'brow': 0.7110501029512697, 'mouth': 0.7110501029512697}
Normalized weights: {'eye': 0.3333333333333333, 'brow': 0.3333333333333333, 'mouth': 0.3333333333333333}

VALIDATION WEIGHT RECOVERY: efficientnet_b0
Validation ROC-AUC: {'eye': 0.7110501029512697, 'brow': 0.7110501029512697, 'mouth': 0.7110501029512697}
Normalized weights: {'eye': 0.3333333333333333, 'brow': 0.3333333333333333, 'mouth': 0.3333333333333333}

VALIDATION WEIGHT RECOVERY: swinv2_texture
Validation ROC-AUC: {'eye': 0.7387096881866455, 'brow': 0.7110501029512697, 'mouth': 0.7110501029512697}
Normalized weights: {'eye': 0.36123806607899933, 'brow': 0.31938096696050033, 'mouth': 0.31938096696050033}


,model_family,region,validation_roc_auc,normalized_weight,source_path,source_field
0,swinv2_tiny,eye,0.71105,0.333333,/content/drive/MyDrive/AISC DeepFake Çalışmala...,model_selection.best_validation_metrics.roc_auc
1,swinv2_tiny,brow,0.71105,0.333333,/content/drive/MyDrive/AISC DeepFake Çalışmala...,model_selection.best_validation_metrics.roc_auc
2,swinv2_tiny,mouth,0.71105,0.333333,/content/drive/MyDrive/AISC DeepFake Çalışmala...,model_selection.best_validation_metrics.roc_auc
3,efficientnet_b0,eye,0.71105,0.333333,/content/drive/MyDrive/AISC DeepFake Çalışmala...,model_selection.best_validation_metrics.roc_auc
4,efficientnet_b0,brow,0.71105,0.333333,/content/drive/MyDrive/AISC DeepFake Çalışmala...,model_selection.best_validation_metrics.roc_auc
5,efficientnet_b0,mouth,0.71105,0.333333,/content/drive/MyDrive/AISC DeepFake Çalışmala...,model_selection.best_validation_metrics.roc_auc
6,swinv2_texture,eye,0.73871,0.361238,/content/drive/MyDrive/AISC DeepFake Çalışmala...,best_finetune_val_auc
7,swinv2_texture,brow,0.71105,0.319381,/content/drive/MyDrive/AISC DeepFake Çalışmala...,model_selection.best_validation_metrics.roc_auc
8,swinv2_texture,mouth,0.71105,0.319381,/content/drive/MyDrive/AISC DeepFake Çalışmala...,model_selection.best_validation_metrics.roc_auc


In [7]:
# ============================================================
# 7) VISUALIZATION HELPERS
# ============================================================

DISPLAY_NAMES = {
    "eye_only": "Eye only",
    "brow_only": "Brow only",
    "mouth_only": "Mouth only",
    "fusion": "Weighted Soft Voting",
}

PROBABILITY_COLUMNS = {
    "eye_only": "p_eye",
    "brow_only": "p_brow",
    "mouth_only": "p_mouth",
    "fusion": "fusion_probability",
}


def plot_weights(
    weights,
    validation_auc,
    family_name,
    figure_dir,
):
    labels = [
        "Eye",
        "Brow",
        "Mouth",
    ]

    regions = [
        "eye",
        "brow",
        "mouth",
    ]

    values = [
        weights[region]
        for region in regions
    ]

    fig, ax = plt.subplots(
        figsize=(9, 6)
    )

    bars = ax.bar(
        labels,
        values,
    )

    ax.set_title(
        f"{family_name} — Validation-Based Fusion Weights",
        fontsize=14,
        fontweight="bold",
        pad=12,
    )

    ax.set_ylabel(
        "Normalized Weight",
        fontsize=11,
    )

    ax.set_ylim(
        0,
        max(
            1.0,
            max(values) * 1.25,
        ),
    )

    ax.grid(
        axis="y",
        alpha=0.25,
    )

    for bar, region, value in zip(
        bars,
        regions,
        values,
    ):
        ax.text(
            bar.get_x()
            + bar.get_width() / 2,
            bar.get_height(),
            (
                f"w={value:.3f}\n"
                f"Val AUC={validation_auc[region]:.3f}"
            ),
            ha="center",
            va="bottom",
            fontsize=10,
        )

    fig.tight_layout()

    save_figure(
        fig,
        figure_dir
        / "validation_based_weights",
    )


def plot_coverage(
    audit,
    family_name,
    figure_dir,
):
    counts = audit[
        "counts"
    ]

    labels = [
        "Eye",
        "Brow",
        "Mouth",
        "Common",
    ]

    values = [
        counts["eye_frames"],
        counts["brow_frames"],
        counts["mouth_frames"],
        counts["common_frames"],
    ]

    fig, ax = plt.subplots(
        figsize=(9, 6)
    )

    bars = ax.bar(
        labels,
        values,
    )

    ax.set_title(
        f"{family_name} — Frame Coverage",
        fontsize=14,
        fontweight="bold",
        pad=12,
    )

    ax.set_ylabel(
        "Number of Upstream Frames",
        fontsize=11,
    )

    ax.grid(
        axis="y",
        alpha=0.25,
    )

    for bar, value in zip(
        bars,
        values,
    ):
        ax.text(
            bar.get_x()
            + bar.get_width() / 2,
            bar.get_height(),
            str(value),
            ha="center",
            va="bottom",
            fontsize=11,
        )

    fig.tight_layout()

    save_figure(
        fig,
        figure_dir
        / "frame_coverage",
    )


def plot_class_distribution(
    dataframe,
    family_name,
    figure_dir,
):
    counts = (
        dataframe[
            "label"
        ]
        .value_counts()
        .reindex(
            [0, 1],
            fill_value=0,
        )
    )

    fig, ax = plt.subplots(
        figsize=(8, 6)
    )

    bars = ax.bar(
        [
            "REAL",
            "FAKE",
        ],
        counts.values,
    )

    ax.set_title(
        f"{family_name} — Common-Set Class Distribution",
        fontsize=14,
        fontweight="bold",
        pad=12,
    )

    ax.set_ylabel(
        "Number of Frames",
        fontsize=11,
    )

    ax.grid(
        axis="y",
        alpha=0.25,
    )

    for bar, value in zip(
        bars,
        counts.values,
    ):
        ax.text(
            bar.get_x()
            + bar.get_width() / 2,
            bar.get_height(),
            str(int(value)),
            ha="center",
            va="bottom",
            fontsize=11,
        )

    fig.tight_layout()

    save_figure(
        fig,
        figure_dir
        / "common_set_class_distribution",
    )


def plot_roc_comparison(
    dataframe,
    family_name,
    figure_dir,
):
    y_true = dataframe[
        "label"
    ].to_numpy()

    fig, ax = plt.subplots(
        figsize=(9, 7)
    )

    for evaluation, column in (
        PROBABILITY_COLUMNS.items()
    ):
        probability = dataframe[
            column
        ].to_numpy()

        fpr, tpr, _ = roc_curve(
            y_true,
            probability,
        )

        auc_value = roc_auc_score(
            y_true,
            probability,
        )

        ax.plot(
            fpr,
            tpr,
            linewidth=2,
            label=(
                f"{DISPLAY_NAMES[evaluation]} "
                f"(AUC={auc_value:.3f})"
            ),
        )

    ax.plot(
        [0, 1],
        [0, 1],
        linestyle="--",
        linewidth=1.5,
        label="Chance",
    )

    ax.set_title(
        f"{family_name} — ROC Comparison",
        fontsize=14,
        fontweight="bold",
        pad=12,
    )

    ax.set_xlabel(
        "False Positive Rate",
        fontsize=11,
    )

    ax.set_ylabel(
        "True Positive Rate",
        fontsize=11,
    )

    ax.legend(
        frameon=True
    )

    ax.grid(
        alpha=0.25
    )

    fig.tight_layout()

    save_figure(
        fig,
        figure_dir
        / "roc_comparison",
    )


def plot_pr_comparison(
    dataframe,
    family_name,
    figure_dir,
):
    y_true = dataframe[
        "label"
    ].to_numpy()

    fig, ax = plt.subplots(
        figsize=(9, 7)
    )

    for evaluation, column in (
        PROBABILITY_COLUMNS.items()
    ):
        probability = dataframe[
            column
        ].to_numpy()

        precision, recall, _ = (
            precision_recall_curve(
                y_true,
                probability,
            )
        )

        ap_value = (
            average_precision_score(
                y_true,
                probability,
            )
        )

        ax.plot(
            recall,
            precision,
            linewidth=2,
            label=(
                f"{DISPLAY_NAMES[evaluation]} "
                f"(AP={ap_value:.3f})"
            ),
        )

    ax.set_title(
        f"{family_name} — Precision–Recall Comparison",
        fontsize=14,
        fontweight="bold",
        pad=12,
    )

    ax.set_xlabel(
        "Recall",
        fontsize=11,
    )

    ax.set_ylabel(
        "Precision",
        fontsize=11,
    )

    ax.legend(
        frameon=True
    )

    ax.grid(
        alpha=0.25
    )

    fig.tight_layout()

    save_figure(
        fig,
        figure_dir
        / "precision_recall_comparison",
    )


def plot_confusion_matrix(
    dataframe,
    family_name,
    figure_dir,
):
    y_true = dataframe[
        "label"
    ].to_numpy()

    predicted = (
        dataframe[
            "fusion_probability"
        ].to_numpy()
        >= DECISION_THRESHOLD
    ).astype(int)

    matrix = confusion_matrix(
        y_true,
        predicted,
        labels=[0, 1],
    )

    fig, ax = plt.subplots(
        figsize=(7, 6)
    )

    image = ax.imshow(
        matrix
    )

    ax.set_xticks(
        [0, 1],
        labels=[
            "REAL",
            "FAKE",
        ],
    )

    ax.set_yticks(
        [0, 1],
        labels=[
            "REAL",
            "FAKE",
        ],
    )

    ax.set_xlabel(
        "Predicted Class",
        fontsize=11,
    )

    ax.set_ylabel(
        "True Class",
        fontsize=11,
    )

    ax.set_title(
        f"{family_name} — Weighted Soft Voting Confusion Matrix",
        fontsize=14,
        fontweight="bold",
        pad=12,
    )

    for row in range(2):
        for column in range(2):
            ax.text(
                column,
                row,
                str(
                    int(
                        matrix[
                            row,
                            column,
                        ]
                    )
                ),
                ha="center",
                va="center",
                fontsize=12,
            )

    fig.colorbar(
        image,
        ax=ax,
    )

    fig.tight_layout()

    save_figure(
        fig,
        figure_dir
        / "fusion_confusion_matrix",
    )


def plot_probability_distributions(
    dataframe,
    family_name,
    figure_dir,
):
    fig, ax = plt.subplots(
        figsize=(10, 7)
    )

    for evaluation, column in (
        PROBABILITY_COLUMNS.items()
    ):
        ax.hist(
            dataframe[
                column
            ].to_numpy(),
            bins=20,
            alpha=0.35,
            label=DISPLAY_NAMES[
                evaluation
            ],
        )

    ax.axvline(
        DECISION_THRESHOLD,
        linestyle="--",
        linewidth=2,
        label=(
            f"Decision Threshold "
            f"({DECISION_THRESHOLD:.2f})"
        ),
    )

    ax.set_title(
        f"{family_name} — Probability Distributions",
        fontsize=14,
        fontweight="bold",
        pad=12,
    )

    ax.set_xlabel(
        "Predicted FAKE Probability",
        fontsize=11,
    )

    ax.set_ylabel(
        "Frame Count",
        fontsize=11,
    )

    ax.legend(
        frameon=True
    )

    ax.grid(
        alpha=0.25
    )

    fig.tight_layout()

    save_figure(
        fig,
        figure_dir
        / "probability_distributions",
    )


def plot_probability_by_class(
    dataframe,
    family_name,
    figure_dir,
):
    real = dataframe.loc[
        dataframe["label"] == 0,
        "fusion_probability",
    ].to_numpy()

    fake = dataframe.loc[
        dataframe["label"] == 1,
        "fusion_probability",
    ].to_numpy()

    fig, ax = plt.subplots(
        figsize=(8, 6)
    )

    ax.boxplot(
        [
            real,
            fake,
        ],
        tick_labels=[
            "REAL",
            "FAKE",
        ],
        showmeans=True,
    )

    ax.axhline(
        DECISION_THRESHOLD,
        linestyle="--",
        linewidth=2,
        label=(
            f"Decision Threshold "
            f"({DECISION_THRESHOLD:.2f})"
        ),
    )

    ax.set_title(
        f"{family_name} — Fusion Probability by True Class",
        fontsize=14,
        fontweight="bold",
        pad=12,
    )

    ax.set_ylabel(
        "Predicted FAKE Probability",
        fontsize=11,
    )

    ax.legend(
        frameon=True
    )

    ax.grid(
        axis="y",
        alpha=0.25,
    )

    fig.tight_layout()

    save_figure(
        fig,
        figure_dir
        / "fusion_probability_by_true_class",
    )


def plot_prediction_correlation(
    dataframe,
    family_name,
    figure_dir,
):
    columns = [
        "p_eye",
        "p_brow",
        "p_mouth",
        "fusion_probability",
    ]

    labels = [
        "Eye",
        "Brow",
        "Mouth",
        "Fusion",
    ]

    correlation = dataframe[
        columns
    ].corr()

    fig, ax = plt.subplots(
        figsize=(8, 7)
    )

    image = ax.imshow(
        correlation.to_numpy(),
        vmin=-1,
        vmax=1,
    )

    ax.set_xticks(
        range(
            len(labels)
        ),
        labels=labels,
        rotation=30,
        ha="right",
    )

    ax.set_yticks(
        range(
            len(labels)
        ),
        labels=labels,
    )

    ax.set_title(
        f"{family_name} — Prediction Correlation",
        fontsize=14,
        fontweight="bold",
        pad=12,
    )

    for row in range(
        len(labels)
    ):
        for column in range(
            len(labels)
        ):
            ax.text(
                column,
                row,
                f"{correlation.iloc[row, column]:.2f}",
                ha="center",
                va="center",
                fontsize=11,
            )

    fig.colorbar(
        image,
        ax=ax,
        label="Pearson Correlation",
    )

    fig.tight_layout()

    save_figure(
        fig,
        figure_dir
        / "prediction_correlation",
    )


def plot_regional_agreement(
    dataframe,
    family_name,
    figure_dir,
):
    votes = pd.DataFrame(
        {
            "Eye": (
                dataframe[
                    "p_eye"
                ]
                >= DECISION_THRESHOLD
            ).astype(int),
            "Brow": (
                dataframe[
                    "p_brow"
                ]
                >= DECISION_THRESHOLD
            ).astype(int),
            "Mouth": (
                dataframe[
                    "p_mouth"
                ]
                >= DECISION_THRESHOLD
            ).astype(int),
        }
    )

    patterns = (
        votes
        .astype(str)
        .agg(
            "-".join,
            axis=1,
        )
        .map(
            {
                "0-0-0": "All REAL",
                "1-1-1": "All FAKE",
                "0-0-1": "Mouth only FAKE",
                "0-1-0": "Brow only FAKE",
                "1-0-0": "Eye only FAKE",
                "0-1-1": "Brow + Mouth FAKE",
                "1-0-1": "Eye + Mouth FAKE",
                "1-1-0": "Eye + Brow FAKE",
            }
        )
    )

    counts = (
        patterns
        .value_counts()
    )

    fig, ax = plt.subplots(
        figsize=(11, 7)
    )

    bars = ax.barh(
        counts.index,
        counts.values,
    )

    ax.set_title(
        f"{family_name} — Regional Decision Agreement",
        fontsize=14,
        fontweight="bold",
        pad=12,
    )

    ax.set_xlabel(
        "Number of Common Frames",
        fontsize=11,
    )

    ax.grid(
        axis="x",
        alpha=0.25,
    )

    for bar, value in zip(
        bars,
        counts.values,
    ):
        ax.text(
            bar.get_width(),
            bar.get_y()
            + bar.get_height() / 2,
            str(int(value)),
            va="center",
            ha="left",
            fontsize=10,
        )

    fig.tight_layout()

    save_figure(
        fig,
        figure_dir
        / "regional_decision_agreement",
    )


def plot_metric_comparison(
    metrics_df,
    family_name,
    figure_dir,
):
    selected = metrics_df[
        [
            "evaluation",
            "roc_auc",
            "pr_auc",
            "balanced_accuracy",
            "f1",
        ]
    ].copy()

    selected[
        "evaluation"
    ] = selected[
        "evaluation"
    ].map(
        DISPLAY_NAMES
    )

    plot_df = (
        selected
        .set_index(
            "evaluation"
        )
    )

    fig, ax = plt.subplots(
        figsize=(11, 7)
    )

    plot_df.plot(
        kind="bar",
        ax=ax,
    )

    ax.set_title(
        f"{family_name} — Regional vs Weighted Fusion Metrics",
        fontsize=14,
        fontweight="bold",
        pad=12,
    )

    ax.set_xlabel(
        ""
    )

    ax.set_ylabel(
        "Score",
        fontsize=11,
    )

    ax.set_ylim(
        0,
        1.05,
    )

    ax.tick_params(
        axis="x",
        rotation=20,
    )

    ax.legend(
        [
            "ROC-AUC",
            "PR-AUC",
            "Balanced Accuracy",
            "F1",
        ],
        frameon=True,
    )

    ax.grid(
        axis="y",
        alpha=0.25,
    )

    fig.tight_layout()

    save_figure(
        fig,
        figure_dir
        / "regional_vs_fusion_metrics",
    )


def create_family_figures(
    aligned_df,
    metrics_df,
    audit,
    weights,
    validation_auc,
    family_name,
    figure_dir,
):
    figure_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    plot_weights(
        weights=weights,
        validation_auc=validation_auc,
        family_name=family_name,
        figure_dir=figure_dir,
    )

    plot_coverage(
        audit=audit,
        family_name=family_name,
        figure_dir=figure_dir,
    )

    plot_class_distribution(
        dataframe=aligned_df,
        family_name=family_name,
        figure_dir=figure_dir,
    )

    plot_roc_comparison(
        dataframe=aligned_df,
        family_name=family_name,
        figure_dir=figure_dir,
    )

    plot_pr_comparison(
        dataframe=aligned_df,
        family_name=family_name,
        figure_dir=figure_dir,
    )

    plot_confusion_matrix(
        dataframe=aligned_df,
        family_name=family_name,
        figure_dir=figure_dir,
    )

    plot_probability_distributions(
        dataframe=aligned_df,
        family_name=family_name,
        figure_dir=figure_dir,
    )

    plot_probability_by_class(
        dataframe=aligned_df,
        family_name=family_name,
        figure_dir=figure_dir,
    )

    plot_prediction_correlation(
        dataframe=aligned_df,
        family_name=family_name,
        figure_dir=figure_dir,
    )

    plot_regional_agreement(
        dataframe=aligned_df,
        family_name=family_name,
        figure_dir=figure_dir,
    )

    plot_metric_comparison(
        metrics_df=metrics_df,
        family_name=family_name,
        figure_dir=figure_dir,
    )

In [8]:
# ============================================================
# 8) WEIGHTED SOFT VOTING — ALL THREE MODEL FAMILIES
# ============================================================

all_metric_rows = []
family_audit_rows = []

for family, cfg in MODEL_FAMILIES.items():
    print("\n" + "=" * 90)
    print(
        f"MODEL FAMILY: {family}"
    )
    print("=" * 90)

    weight_config = (
        WEIGHT_CONFIGS[
            family
        ]
    )

    weights = (
        weight_config[
            "weights"
        ]
    )

    validation_auc = (
        weight_config[
            "validation_auc"
        ]
    )

    aligned, audit = (
        align_three(
            cfg["eye_test"],
            cfg["brow_test"],
            cfg["mouth_test"],
        )
    )

    aligned[
        "fusion_probability"
    ] = (
        weights["eye"]
        * aligned["p_eye"]
        + weights["brow"]
        * aligned["p_brow"]
        + weights["mouth"]
        * aligned["p_mouth"]
    )

    fusion_values = (
        aligned[
            "fusion_probability"
        ]
        .to_numpy()
    )

    if not np.isfinite(
        fusion_values
    ).all():
        raise RuntimeError(
            f"{family}: NaN/Inf in weighted fusion probabilities."
        )

    if (
        (
            aligned[
                "fusion_probability"
            ]
            < 0
        )
        | (
            aligned[
                "fusion_probability"
            ]
            > 1
        )
    ).any():
        raise RuntimeError(
            f"{family}: weighted fusion probability outside [0, 1]."
        )

    family_dir = (
        RUN_DIR
        / family
    )

    figure_dir = (
        family_dir
        / "figures"
    )

    metric_rows = []

    for evaluation, probability_column in (
        PROBABILITY_COLUMNS.items()
    ):
        metrics = (
            compute_metrics(
                aligned["label"],
                aligned[
                    probability_column
                ],
                threshold=DECISION_THRESHOLD,
            )
        )

        row = {
            "model_family": family,
            "method": METHOD_NAME,
            "evaluation": evaluation,
            "threshold_source": (
                "fixed_predefined_0.50"
            ),
            "weight_source": (
                "validation_roc_auc_only"
            ),
            "weight_eye": float(
                weights["eye"]
            ),
            "weight_brow": float(
                weights["brow"]
            ),
            "weight_mouth": float(
                weights["mouth"]
            ),
            "validation_auc_eye": float(
                validation_auc["eye"]
            ),
            "validation_auc_brow": float(
                validation_auc["brow"]
            ),
            "validation_auc_mouth": float(
                validation_auc["mouth"]
            ),
            **metrics,
        }

        metric_rows.append(
            row
        )

        all_metric_rows.append(
            row
        )

    family_metrics = pd.DataFrame(
        metric_rows
    )

    if len(aligned) != audit[
        "counts"
    ][
        "common_frames"
    ]:
        raise RuntimeError(
            f"{family}: alignment accounting mismatch."
        )

    if aligned[
        "fusion_key"
    ].duplicated().any():
        raise RuntimeError(
            f"{family}: duplicate fusion_key after alignment."
        )

    if not set(
        aligned[
            "label"
        ].unique()
    ).issubset(
        {
            0,
            1,
        }
    ):
        raise RuntimeError(
            f"{family}: invalid labels after alignment."
        )

    atomic_write_csv(
        aligned,
        family_dir
        / "predictions"
        / "aligned_test_predictions.csv",
    )

    atomic_write_csv(
        family_metrics,
        family_dir
        / "metrics"
        / "test_metrics.csv",
    )

    atomic_write_json(
        {
            "run_id": RUN_ID,
            "method": METHOD_NAME,
            "model_family": family,
            "decision_threshold": DECISION_THRESHOLD,
            "threshold_source": (
                "fixed_predefined_0.50"
            ),
            "frame_aggregation": FRAME_AGGREGATION,
            "weight_config": weight_config,
            "alignment_audit": audit,
            "fusion_metrics": (
                family_metrics.loc[
                    family_metrics[
                        "evaluation"
                    ]
                    == "fusion"
                ]
                .iloc[0]
                .to_dict()
            ),
        },
        family_dir
        / "audit"
        / "run_audit.json",
    )

    create_family_figures(
        aligned_df=aligned,
        metrics_df=family_metrics,
        audit=audit,
        weights=weights,
        validation_auc=validation_auc,
        family_name=family,
        figure_dir=figure_dir,
    )

    family_audit_rows.append(
        {
            "model_family": family,
            **audit["counts"],
            "eye_multi_roi_frames": (
                audit["eye"][
                    "multi_roi_frames"
                ]
            ),
            "brow_multi_roi_frames": (
                audit["brow"][
                    "multi_roi_frames"
                ]
            ),
            "mouth_multi_roi_frames": (
                audit["mouth"][
                    "multi_roi_frames"
                ]
            ),
            "weight_eye": float(
                weights["eye"]
            ),
            "weight_brow": float(
                weights["brow"]
            ),
            "weight_mouth": float(
                weights["mouth"]
            ),
        }
    )

    print(
        f"Common frames: "
        f"{audit['counts']['common_frames']} "
        f"(REAL="
        f"{audit['counts']['real_common_frames']}, "
        f"FAKE="
        f"{audit['counts']['fake_common_frames']})"
    )

    print(
        "Weights:",
        weights,
    )

    display(
        family_metrics[
            [
                "evaluation",
                "n",
                "accuracy",
                "balanced_accuracy",
                "precision",
                "recall",
                "specificity",
                "f1",
                "roc_auc",
                "pr_auc",
            ]
        ]
    )


all_metrics = pd.DataFrame(
    all_metric_rows
)

family_audit = pd.DataFrame(
    family_audit_rows
)

atomic_write_csv(
    all_metrics,
    RUN_DIR
    / "metrics"
    / "all_model_families_metrics.csv",
)

atomic_write_csv(
    family_audit,
    RUN_DIR
    / "audit"
    / "family_alignment_summary.csv",
)

print(
    "\nAll three model families completed."
)


MODEL FAMILY: swinv2_tiny
EYE   | Rows=302 | Frames=292 | Key=prediction_column
BROW  | Rows=196 | Frames=196 | Key=prediction_column
MOUTH | Rows=302 | Frames=292 | Key=prediction_column
Common frames: 196 (REAL=101, FAKE=95)
Weights: {'eye': 0.3333333333333333, 'brow': 0.3333333333333333, 'mouth': 0.3333333333333333}


,evaluation,n,accuracy,balanced_accuracy,precision,recall,specificity,f1,roc_auc,pr_auc
0,eye_only,196,0.744898,0.743408,0.758621,0.694737,0.792079,0.725275,0.782908,0.786848
1,brow_only,196,0.668367,0.667587,0.663043,0.642105,0.693069,0.652406,0.681501,0.654312
2,mouth_only,196,0.806122,0.804065,0.843373,0.736842,0.871287,0.786517,0.880250,0.870051
3,fusion,196,0.770408,0.768161,0.804878,0.694737,0.841584,0.745763,0.858676,0.865218



MODEL FAMILY: efficientnet_b0
EYE   | Rows=302 | Frames=292 | Key=companion_metadata
BROW  | Rows=196 | Frames=196 | Key=companion_metadata
MOUTH | Rows=302 | Frames=292 | Key=prediction_column
Common frames: 196 (REAL=101, FAKE=95)
Weights: {'eye': 0.3333333333333333, 'brow': 0.3333333333333333, 'mouth': 0.3333333333333333}


,evaluation,n,accuracy,balanced_accuracy,precision,recall,specificity,f1,roc_auc,pr_auc
0,eye_only,196,0.673469,0.673476,0.659794,0.673684,0.673267,0.666667,0.731318,0.728882
1,brow_only,196,0.551020,0.552788,0.532110,0.610526,0.495050,0.568627,0.587389,0.576110
2,mouth_only,196,0.755102,0.754560,0.752688,0.736842,0.772277,0.744681,0.833351,0.837753
3,fusion,196,0.724490,0.723919,0.720430,0.705263,0.742574,0.712766,0.801668,0.805531



MODEL FAMILY: swinv2_texture
EYE   | Rows=302 | Frames=292 | Key=prediction_column
BROW  | Rows=196 | Frames=196 | Key=prediction_column
MOUTH | Rows=302 | Frames=292 | Key=prediction_column
Common frames: 196 (REAL=101, FAKE=95)
Weights: {'eye': 0.36123806607899933, 'brow': 0.31938096696050033, 'mouth': 0.31938096696050033}


,evaluation,n,accuracy,balanced_accuracy,precision,recall,specificity,f1,roc_auc,pr_auc
0,eye_only,196,0.642857,0.640021,0.658228,0.547368,0.732673,0.597701,0.675560,0.644614
1,brow_only,196,0.622449,0.619281,0.636364,0.515789,0.722772,0.569767,0.668786,0.670791
2,mouth_only,196,0.724490,0.723293,0.730337,0.684211,0.762376,0.706522,0.789682,0.751422
3,fusion,196,0.698980,0.696978,0.714286,0.631579,0.762376,0.670391,0.751537,0.733694



All three model families completed.


In [9]:
# ============================================================
# 9) CROSS-FAMILY VISUAL COMPARISONS
# ============================================================

overall_figure_dir = (
    RUN_DIR
    / "figures"
)

overall_figure_dir.mkdir(
    parents=True,
    exist_ok=True,
)

fusion_only = (
    all_metrics.loc[
        all_metrics[
            "evaluation"
        ]
        == "fusion"
    ]
    .copy()
    .sort_values(
        "model_family"
    )
)


# ------------------------------------------------------------
# A) Weighted fusion metric comparison
# ------------------------------------------------------------

metric_columns = [
    "roc_auc",
    "pr_auc",
    "balanced_accuracy",
    "f1",
]

plot_data = (
    fusion_only[
        [
            "model_family",
            *metric_columns,
        ]
    ]
    .set_index(
        "model_family"
    )
)

fig, ax = plt.subplots(
    figsize=(12, 7)
)

plot_data.plot(
    kind="bar",
    ax=ax,
)

ax.set_title(
    "Weighted Soft Voting — Model Family Comparison",
    fontsize=14,
    fontweight="bold",
    pad=12,
)

ax.set_xlabel(
    "Model Family",
    fontsize=11,
)

ax.set_ylabel(
    "Score",
    fontsize=11,
)

ax.set_ylim(
    0,
    1.05,
)

ax.tick_params(
    axis="x",
    rotation=15,
)

ax.legend(
    [
        "ROC-AUC",
        "PR-AUC",
        "Balanced Accuracy",
        "F1",
    ],
    frameon=True,
)

ax.grid(
    axis="y",
    alpha=0.25,
)

fig.tight_layout()

save_figure(
    fig,
    overall_figure_dir
    / "weighted_soft_voting_model_family_comparison",
)


# ------------------------------------------------------------
# B) Weight comparison across model families
# ------------------------------------------------------------

weight_plot = (
    family_audit[
        [
            "model_family",
            "weight_eye",
            "weight_brow",
            "weight_mouth",
        ]
    ]
    .set_index(
        "model_family"
    )
)

fig, ax = plt.subplots(
    figsize=(12, 7)
)

weight_plot.plot(
    kind="bar",
    ax=ax,
)

ax.set_title(
    "Validation-Based Region Weights Across Model Families",
    fontsize=14,
    fontweight="bold",
    pad=12,
)

ax.set_xlabel(
    "Model Family",
    fontsize=11,
)

ax.set_ylabel(
    "Normalized Weight",
    fontsize=11,
)

ax.set_ylim(
    0,
    1.0,
)

ax.tick_params(
    axis="x",
    rotation=15,
)

ax.legend(
    [
        "Eye",
        "Brow",
        "Mouth",
    ],
    frameon=True,
)

ax.grid(
    axis="y",
    alpha=0.25,
)

fig.tight_layout()

save_figure(
    fig,
    overall_figure_dir
    / "region_weights_across_model_families",
)


# ------------------------------------------------------------
# C) Frame coverage comparison
# ------------------------------------------------------------

coverage_plot = (
    family_audit[
        [
            "model_family",
            "eye_frames",
            "brow_frames",
            "mouth_frames",
            "common_frames",
        ]
    ]
    .set_index(
        "model_family"
    )
)

fig, ax = plt.subplots(
    figsize=(12, 7)
)

coverage_plot.plot(
    kind="bar",
    ax=ax,
)

ax.set_title(
    "Model Family — Available and Common Frame Counts",
    fontsize=14,
    fontweight="bold",
    pad=12,
)

ax.set_xlabel(
    "Model Family",
    fontsize=11,
)

ax.set_ylabel(
    "Number of Upstream Frames",
    fontsize=11,
)

ax.tick_params(
    axis="x",
    rotation=15,
)

ax.legend(
    [
        "Eye",
        "Brow",
        "Mouth",
        "Common",
    ],
    frameon=True,
)

ax.grid(
    axis="y",
    alpha=0.25,
)

fig.tight_layout()

save_figure(
    fig,
    overall_figure_dir
    / "model_family_frame_coverage_comparison",
)


# ------------------------------------------------------------
# D) Fusion gain vs best single region
# ------------------------------------------------------------

gain_rows = []

for family in (
    all_metrics[
        "model_family"
    ]
    .unique()
):
    subset = all_metrics[
        all_metrics[
            "model_family"
        ]
        == family
    ]

    single = subset[
        subset[
            "evaluation"
        ].isin(
            [
                "eye_only",
                "brow_only",
                "mouth_only",
            ]
        )
    ]

    fusion = subset[
        subset[
            "evaluation"
        ]
        == "fusion"
    ].iloc[0]

    gain_rows.append(
        {
            "model_family": family,
            "roc_auc_gain_vs_best_single": float(
                fusion["roc_auc"]
                - single[
                    "roc_auc"
                ].max()
            ),
            "f1_gain_vs_best_single": float(
                fusion["f1"]
                - single[
                    "f1"
                ].max()
            ),
        }
    )

gain_df = pd.DataFrame(
    gain_rows
)

atomic_write_csv(
    gain_df,
    RUN_DIR
    / "metrics"
    / "fusion_gain_vs_best_single_region.csv",
)

fig, ax = plt.subplots(
    figsize=(11, 7)
)

gain_df.set_index(
    "model_family"
).plot(
    kind="bar",
    ax=ax,
)

ax.axhline(
    0,
    linestyle="--",
    linewidth=1.5,
)

ax.set_title(
    "Weighted Soft Voting — Gain vs Best Single Region",
    fontsize=14,
    fontweight="bold",
    pad=12,
)

ax.set_xlabel(
    "Model Family",
    fontsize=11,
)

ax.set_ylabel(
    "Absolute Metric Difference",
    fontsize=11,
)

ax.tick_params(
    axis="x",
    rotation=15,
)

ax.legend(
    [
        "ROC-AUC Gain",
        "F1 Gain",
    ],
    frameon=True,
)

ax.grid(
    axis="y",
    alpha=0.25,
)

fig.tight_layout()

save_figure(
    fig,
    overall_figure_dir
    / "fusion_gain_vs_best_single_region",
)

display(
    fusion_only
)

display(
    gain_df
)

,model_family,method,evaluation,threshold_source,weight_source,weight_eye,weight_brow,weight_mouth,validation_auc_eye,validation_auc_brow,...,precision,recall,specificity,f1,roc_auc,pr_auc,tn,fp,fn,tp
7,efficientnet_b0,02_weighted_soft_voting,fusion,fixed_predefined_0.50,validation_roc_auc_only,0.333333,0.333333,0.333333,0.71105,0.71105,...,0.720430,0.705263,0.742574,0.712766,0.801668,0.805531,75,26,28,67
11,swinv2_texture,02_weighted_soft_voting,fusion,fixed_predefined_0.50,validation_roc_auc_only,0.361238,0.319381,0.319381,0.73871,0.71105,...,0.714286,0.631579,0.762376,0.670391,0.751537,0.733694,77,24,35,60
3,swinv2_tiny,02_weighted_soft_voting,fusion,fixed_predefined_0.50,validation_roc_auc_only,0.333333,0.333333,0.333333,0.71105,0.71105,...,0.804878,0.694737,0.841584,0.745763,0.858676,0.865218,85,16,29,66


,model_family,roc_auc_gain_vs_best_single,f1_gain_vs_best_single
0,swinv2_tiny,-0.021574,-0.040754
1,efficientnet_b0,-0.031683,-0.031915
2,swinv2_texture,-0.038145,-0.036131


In [10]:
# ============================================================
# 10) FINAL QUALITY GATES + OUTPUT MANIFEST
# ============================================================

required_metric_columns = [
    "accuracy",
    "balanced_accuracy",
    "precision",
    "recall",
    "specificity",
    "f1",
    "roc_auc",
    "pr_auc",
]

missing_metric_columns = [
    column
    for column in required_metric_columns
    if column not in all_metrics.columns
]

if missing_metric_columns:
    raise RuntimeError(
        f"Missing metric columns: "
        f"{missing_metric_columns}"
    )

numeric_metrics = (
    all_metrics[
        required_metric_columns
    ]
    .apply(
        pd.to_numeric,
        errors="coerce",
    )
)

if not np.isfinite(
    numeric_metrics.to_numpy()
).all():
    raise RuntimeError(
        "NaN/Inf found in final metrics."
    )

for column in (
    required_metric_columns
):
    invalid = (
        (all_metrics[column] < 0)
        | (all_metrics[column] > 1)
    )

    if invalid.any():
        raise RuntimeError(
            f"{column}: final metric outside [0, 1]."
        )


expected_evaluations = {
    "eye_only",
    "brow_only",
    "mouth_only",
    "fusion",
}

for family in MODEL_FAMILIES:
    actual = set(
        all_metrics.loc[
            all_metrics[
                "model_family"
            ]
            == family,
            "evaluation",
        ]
    )

    if actual != expected_evaluations:
        raise RuntimeError(
            f"{family}: expected evaluations "
            f"{expected_evaluations}, "
            f"found {actual}"
        )


for family, weight_config in (
    WEIGHT_CONFIGS.items()
):
    weights = weight_config[
        "weights"
    ]

    if not math.isclose(
        sum(
            weights.values()
        ),
        1.0,
        rel_tol=1e-9,
        abs_tol=1e-9,
    ):
        raise RuntimeError(
            f"{family}: final weights do not sum to 1."
        )

    for region, source in (
        weight_config[
            "validation_sources"
        ].items()
    ):
        source_text = (
            str(
                source[
                    "source_path"
                ]
            )
            .lower()
        )

        field_text = (
            str(
                source[
                    "source_field"
                ]
            )
            .lower()
        )

        # Fail if the selected source explicitly looks like a test metric.
        if (
            source_text != "manual_override"
            and (
                "test_metric"
                in source_text
                or "test_auc"
                in field_text
            )
        ):
            raise RuntimeError(
                f"{family}/{region}: possible TEST leakage "
                "detected in validation-weight source."
            )


manifest_rows = []

for path in sorted(
    RUN_DIR.rglob("*")
):
    if not path.is_file():
        continue

    record = {
        "relative_path": str(
            path.relative_to(
                RUN_DIR
            )
        ),
        "size_bytes": int(
            path.stat().st_size
        ),
        "suffix": (
            path.suffix
            .lower()
        ),
    }

    if (
        path.suffix.lower()
        == ".png"
    ):
        with Image.open(
            path
        ) as image:
            record[
                "width_px"
            ] = int(
                image.width
            )

            record[
                "height_px"
            ] = int(
                image.height
            )

            if min(
                image.size
            ) < MIN_FIGURE_SHORT_EDGE_PX:
                raise RuntimeError(
                    "Figure below minimum pixel size: "
                    f"{path} -> {image.size}"
                )

    manifest_rows.append(
        record
    )


manifest = pd.DataFrame(
    manifest_rows
)

atomic_write_csv(
    manifest,
    RUN_DIR
    / "output_manifest.csv",
)


final_summary = {
    "run_id": RUN_ID,
    "method": METHOD_NAME,
    "seed": SEED,
    "decision_threshold": DECISION_THRESHOLD,
    "threshold_source": (
        "fixed_predefined_0.50"
    ),
    "weight_source": (
        "validation_roc_auc_only"
    ),
    "weight_formula": (
        "max(validation_roc_auc - 0.5, epsilon), "
        "normalized to sum=1"
    ),
    "frame_aggregation": FRAME_AGGREGATION,
    "model_families_completed": sorted(
        all_metrics[
            "model_family"
        ]
        .unique()
        .tolist()
    ),
    "metric_rows": int(
        len(all_metrics)
    ),
    "output_files_before_manifest": int(
        len(manifest_rows)
    ),
    "quality_gates": "PASSED",
}

atomic_write_json(
    final_summary,
    RUN_DIR
    / "run_summary.json",
)


print("=" * 90)
print(
    "FINAL QUALITY GATES: PASSED"
)
print("=" * 90)
print(
    f"Run ID : {RUN_ID}"
)
print(
    f"Output : {RUN_DIR}"
)
print(
    "PNG figures:",
    int(
        (
            manifest[
                "suffix"
            ]
            == ".png"
        )
        .sum()
    ),
)
print(
    "SVG figures:",
    int(
        (
            manifest[
                "suffix"
            ]
            == ".svg"
        )
        .sum()
    ),
)

display(
    all_metrics.sort_values(
        [
            "model_family",
            "evaluation",
        ]
    )
)

FINAL QUALITY GATES: PASSED
Run ID : 20260809_163157_934154_weighted_soft_voting_seed42
Output : /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deney 1/Kader/Deney 1/Sonuçlar/Fusion_Experiments/02_weighted_soft_voting/20260809_163157_934154_weighted_soft_voting_seed42
PNG figures: 37
SVG figures: 37


,model_family,method,evaluation,threshold_source,weight_source,weight_eye,weight_brow,weight_mouth,validation_auc_eye,validation_auc_brow,...,precision,recall,specificity,f1,roc_auc,pr_auc,tn,fp,fn,tp
5,efficientnet_b0,02_weighted_soft_voting,brow_only,fixed_predefined_0.50,validation_roc_auc_only,0.333333,0.333333,0.333333,0.71105,0.71105,...,0.532110,0.610526,0.495050,0.568627,0.587389,0.576110,50,51,37,58
4,efficientnet_b0,02_weighted_soft_voting,eye_only,fixed_predefined_0.50,validation_roc_auc_only,0.333333,0.333333,0.333333,0.71105,0.71105,...,0.659794,0.673684,0.673267,0.666667,0.731318,0.728882,68,33,31,64
7,efficientnet_b0,02_weighted_soft_voting,fusion,fixed_predefined_0.50,validation_roc_auc_only,0.333333,0.333333,0.333333,0.71105,0.71105,...,0.720430,0.705263,0.742574,0.712766,0.801668,0.805531,75,26,28,67
6,efficientnet_b0,02_weighted_soft_voting,mouth_only,fixed_predefined_0.50,validation_roc_auc_only,0.333333,0.333333,0.333333,0.71105,0.71105,...,0.752688,0.736842,0.772277,0.744681,0.833351,0.837753,78,23,25,70
9,swinv2_texture,02_weighted_soft_voting,brow_only,fixed_predefined_0.50,validation_roc_auc_only,0.361238,0.319381,0.319381,0.73871,0.71105,...,0.636364,0.515789,0.722772,0.569767,0.668786,0.670791,73,28,46,49
8,swinv2_texture,02_weighted_soft_voting,eye_only,fixed_predefined_0.50,validation_roc_auc_only,0.361238,0.319381,0.319381,0.73871,0.71105,...,0.658228,0.547368,0.732673,0.597701,0.675560,0.644614,74,27,43,52
11,swinv2_texture,02_weighted_soft_voting,fusion,fixed_predefined_0.50,validation_roc_auc_only,0.361238,0.319381,0.319381,0.73871,0.71105,...,0.714286,0.631579,0.762376,0.670391,0.751537,0.733694,77,24,35,60
10,swinv2_texture,02_weighted_soft_voting,mouth_only,fixed_predefined_0.50,validation_roc_auc_only,0.361238,0.319381,0.319381,0.73871,0.71105,...,0.730337,0.684211,0.762376,0.706522,0.789682,0.751422,77,24,30,65
1,swinv2_tiny,02_weighted_soft_voting,brow_only,fixed_predefined_0.50,validation_roc_auc_only,0.333333,0.333333,0.333333,0.71105,0.71105,...,0.663043,0.642105,0.693069,0.652406,0.681501,0.654312,70,31,34,61
0,swinv2_tiny,02_weighted_soft_voting,eye_only,fixed_predefined_0.50,validation_roc_auc_only,0.333333,0.333333,0.333333,0.71105,0.71105,...,0.758621,0.694737,0.792079,0.725275,0.782908,0.786848,80,21,29,66


## Output structure

Her çalıştırmada yeni bir `RUN_ID` klasörü oluşturulur; eski sonuçların üstüne
yazılmaz.

```text
Fusion_Experiments/
└── 02_weighted_soft_voting/
    └── <RUN_ID>/
        ├── environment.json
        ├── run_summary.json
        ├── output_manifest.csv
        ├── audit/
        │   ├── configured_prediction_paths.csv
        │   ├── validation_weight_sources.csv
        │   ├── validation_weight_config.json
        │   └── family_alignment_summary.csv
        ├── metrics/
        │   ├── all_model_families_metrics.csv
        │   └── fusion_gain_vs_best_single_region.csv
        ├── figures/
        │   ├── weighted_soft_voting_model_family_comparison.*
        │   ├── region_weights_across_model_families.*
        │   ├── model_family_frame_coverage_comparison.*
        │   └── fusion_gain_vs_best_single_region.*
        └── <model_family>/
            ├── audit/run_audit.json
            ├── predictions/aligned_test_predictions.csv
            ├── metrics/test_metrics.csv
            └── figures/
                ├── validation_based_weights.*
                ├── frame_coverage.*
                ├── common_set_class_distribution.*
                ├── roc_comparison.*
                ├── precision_recall_comparison.*
                ├── fusion_confusion_matrix.*
                ├── probability_distributions.*
                ├── fusion_probability_by_true_class.*
                ├── prediction_correlation.*
                ├── regional_decision_agreement.*
                └── regional_vs_fusion_metrics.*
```

`*` = both PNG (600 DPI) and SVG.